# 00 — Installation and API overview

This notebook establishes a reproducible RocqiPath environment and
introduces the refactored public API. It is intentionally lightweight:
you can run it before installing OpenSlide, libvips, VALIS, TIAToolbox,
or the cell-counting dependencies.

**Learning goals**

1. Verify that Jupyter is using Python 3.10 or 3.11.
2. Select only the optional extras required by your workflow.
3. Import public APIs from their canonical subpackages.
4. create, inspect, serialize, and reload typed configurations.
5. Understand RocqiPath's shallow output layout.


## Installation

Run these commands in a terminal from the repository root, then restart
the Jupyter kernel:

```bash
python -m pip install -e .
python -m pip install jupyterlab
```

Add capabilities as needed:

```bash
python -m pip install -e ".[extraction]"
python -m pip install -e ".[orb]"
python -m pip install -e ".[valis]"
python -m pip install -e ".[stain]"
python -m pip install -e ".[cellcount]"
python -m pip install -e ".[viz]"
```

Extras can be combined, for example
`python -m pip install -e ".[extraction,orb,cellcount,viz]"`.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    '''Find the RocqiPath repository whether Jupyter starts at root or how_to_use.'''
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "src" / "rocqipath"
        ).is_dir():
            return candidate
    raise FileNotFoundError(
        "RocqiPath repository not found. Start Jupyter inside the cloned repository."
    )


PROJECT_ROOT = find_project_root()
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

DATA_ROOT = PROJECT_ROOT / "data"
RESULTS_ROOT = PROJECT_ROOT / "results"

print(f"Project : {PROJECT_ROOT}")
print(f"Data    : {DATA_ROOT}")
print(f"Results : {RESULTS_ROOT}")


In [ ]:
import importlib.util
import platform

import rocqipath

print(f"Python          : {platform.python_version()}")
print(f"Python binary   : {sys.executable}")
print(f"RocqiPath       : {rocqipath.__version__}")

supported = sys.version_info[:2] in {(3, 10), (3, 11)}
print(f"Supported Python: {supported}")
if not supported:
    print("WARNING: create a 64-bit Python 3.10 or 3.11 environment before WSI work.")


## Optional dependency preflight

`AVAILABLE` below reports Python modules only. OpenSlide and libvips also
require native runtimes. A Python module can therefore be discoverable
while its native library is still missing.


In [ ]:
MODULES = {
    "Pillow": "PIL",
    "NumPy": "numpy",
    "OpenCV": "cv2",
    "OpenSlide": "openslide",
    "pyvips": "pyvips",
    "matplotlib": "matplotlib",
    "scikit-image": "skimage",
    "TIAToolbox": "tiatoolbox",
    "VALIS": "valis",
}

availability = {
    label: importlib.util.find_spec(module_name) is not None
    for label, module_name in MODULES.items()
}
for label, available in availability.items():
    print(f"{label:14s}: {'AVAILABLE' if available else 'not installed'}")


## Public imports

New code should import workflow objects from their feature subpackage:

- `rocqipath.registration` — alignment
- `rocqipath.extraction` — WSI/TMA/patch extraction
- `rocqipath.stain` — stain normalization
- `rocqipath.analysis` — cell counting
- `rocqipath.visualization` — QC and overlays
- `rocqipath.config` — all typed configs
- `rocqipath.core` — slide reading, magnification, output layout, logging

Historical flat modules remain compatibility façades, but they should not
be used in new notebooks.


In [ ]:
from rocqipath.config import AlignmentConfig, CellCountingConfig
from rocqipath.core import OutputLayout

alignment_cfg = AlignmentConfig(
    input_dir=str(DATA_ROOT / "pairs"),
    output_dir=str(RESULTS_ROOT),
    pair_folders=["CD8"],
    reference_name="he",
    moving_name="cd8",
    alignment_method="orb",
    target_magnification=20.0,
    qc_enabled=True,
    dry_run=True,
)

cell_cfg = CellCountingConfig(
    output_dir=str(RESULTS_ROOT),
    target_magnification=20.0,
    patch_size=512,
    tissue_threshold=0.10,
    min_cell_area=50,
)

print("Alignment config:")
for label, value in alignment_cfg.describe():
    print(f"  {label:32s} {value}")

print("\nCell-count config:")
for label, value in cell_cfg.describe():
    print(f"  {label:32s} {value}")


## Save configuration with the analysis

Typed configs inherit `to_dict()` and `from_dict()`. Saving the exact
configuration next to results makes an experiment reproducible and makes
later parameter comparisons much easier.


In [ ]:
import json

config_json = json.dumps(alignment_cfg.to_dict(), indent=2)
print(config_json)

restored_cfg = AlignmentConfig.from_dict(json.loads(config_json))
assert restored_cfg.to_dict() == alignment_cfg.to_dict()
print("\nRound-trip successful.")


In [ ]:
layout = OutputLayout(RESULTS_ROOT)

expected_paths = {
    "alignment module": layout.module_dir("alignment", create=False),
    "one alignment case": layout.item_dir(
        "alignment", "Sample_0001_cd8", create=False
    ),
    "cell-count module": layout.module_dir("cell_counting", create=False),
}

for label, path in expected_paths.items():
    print(f"{label:20s} -> {path}")


## Reproducible notebook checklist

- Keep raw WSIs read-only.
- Use one `RESULTS_ROOT` and let RocqiPath create module/case folders.
- Use physical magnification (`20.0`) everywhere.
- Set scanner-specific source magnification only when metadata is absent.
- Save every config with the results.
- Start with discovery/dry-run cells before processing an entire cohort.
- Keep long-running cells behind an explicit `RUN_*` switch.

Continue with notebook **01** to verify real slide access and magnification.
